In [11]:
# ── BIDIRECTIONAL LSTM  ────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
import time
import matplotlib.pyplot as plt

# Set device
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(" Using MPS")
else:
    device = torch.device("cpu")
    print(" Using CPU")

# Load your FULL LENGTH data
DATA_DIR = Path("data/processed")
data = np.load(DATA_DIR / "sequences.npz")
X_train = data['X_train']  # (956, 986, 129)
X_val = data['X_val']
X_test = data['X_test']
y_val = data['y_val']
y_test = data['y_test']

print(f"Data shapes:")
print(f"  Train: {X_train.shape} (FULL length: {X_train.shape[1]} timesteps)")
print(f"  Val:   {X_val.shape}")
print(f"  Test:  {X_test.shape}")

 Using MPS
Data shapes:
  Train: (956, 986, 129) (FULL length: 986 timesteps)
  Val:   (582, 986, 129)
  Test:  (584, 986, 129)


In [12]:
# ── BIDIRECTIONAL LSTM FOR FULL LENGTH (Regularized to prevent overfitting) ──
class BidirectionalLSTMFull(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=16, dropout=0.3):
        """
        Bidirectional LSTM optimized for long sequences (986 timesteps)
        - Smaller hidden_dim to prevent overfitting
        - Higher dropout for regularization
        - Single layer bidirectional (2 layers would overfit)
        """
        super().__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        
        # Encoder: Single layer bidirectional
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,  # Single layer for long sequences
            batch_first=True,
            bidirectional=True,
            dropout=0
        )
        
        # Project bidirectional output to latent space
        self.fc_encode = nn.Linear(hidden_dim * 2, latent_dim)
        
        # Decoder: Regular LSTM
        self.fc_decode = nn.Linear(latent_dim, hidden_dim)
        
        self.decoder = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            dropout=0
        )
        
        self.output = nn.Linear(hidden_dim, input_dim)
        
        # Regularization
        self.dropout = nn.Dropout(dropout)
        
    def encode(self, x):
        # Bidirectional encoding
        lstm_out, (hidden, cell) = self.encoder(x)
        
        # For single layer bidirectional:
        # lstm_out shape: (batch, seq_len, hidden_dim*2)
        # Take last forward and first backward
        last_forward = lstm_out[:, -1, :self.hidden_dim]
        last_backward = lstm_out[:, 0, self.hidden_dim:]
        
        # Concatenate
        combined = torch.cat([last_forward, last_backward], dim=1)
        
        # Project to latent space
        latent = self.fc_encode(combined)
        
        return latent
    
    def decode(self, latent, seq_len):
        # Expand latent to sequence length
        latent_expanded = latent.unsqueeze(1).repeat(1, seq_len, 1)
        
        # Project to hidden dimension
        hidden = self.fc_decode(latent_expanded)
        hidden = self.dropout(hidden)
        
        # Decode with LSTM
        lstm_out, _ = self.decoder(hidden)
        
        # Generate reconstruction
        reconstruction = self.output(lstm_out)
        
        return reconstruction
    
    def forward(self, x):
        seq_len = x.shape[1]
        latent = self.encode(x)
        reconstruction = self.decode(latent, seq_len)
        return reconstruction, latent

# Initialize model
input_dim = X_train.shape[2]
model = BidirectionalLSTMFull(
    input_dim=input_dim,
    hidden_dim=64,      # Smaller than before (was 128)
    latent_dim=16,      # Smaller latent space
    dropout=0.3         # More dropout for regularization
).to(device)

print(f"Model: Bidirectional LSTM (Full Length)")
print(f"  Input dim: {input_dim}")
print(f"  Hidden dim: 64 (bidirectional → 128 effective)")
print(f"  Latent dim: 16")
print(f"  Dropout: 0.3")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Model: Bidirectional LSTM (Full Length)
  Input dim: 129
  Hidden dim: 64 (bidirectional → 128 effective)
  Latent dim: 16
  Dropout: 0.3
  Total parameters: 144,657


In [13]:
# ── TRAINING WITH STRONGER REGULARIZATION ─────────────────────────────────
# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train).to(device)
X_val_tensor = torch.FloatTensor(X_val).to(device)

# DataLoader
batch_size = 16
train_dataset = TensorDataset(X_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)  # Lower learning rate
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

print(f"Batch size: {batch_size}")
print(f"Learning rate: 0.0003")
print(f"Training batches: {len(train_loader)}")

Batch size: 16
Learning rate: 0.0003
Training batches: 60


In [14]:
# ── TRAINING LOOP ─────────────────────────────────────────────────────────
num_epochs = 50
best_val_loss = float('inf')
patience = 7
patience_counter = 0

train_losses = []
val_losses = []
val_aucs = []

print("\n" + "="*60)
print("TRAINING BIDIRECTIONAL LSTM (FULL LENGTH 986)")
print("="*60)

start_time = time.time()

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0
    for batch in train_loader:
        X_batch = batch[0]
        
        optimizer.zero_grad()
        reconstruction, _ = model(X_batch)
        loss = criterion(reconstruction, X_batch)
        loss.backward()
        
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation
    model.eval()
    val_loss = 0
    val_errors = []
    
    with torch.no_grad():
        for i in range(0, len(X_val_tensor), batch_size):
            batch = X_val_tensor[i:i+batch_size]
            reconstruction, _ = model(batch)
            loss = criterion(reconstruction, batch)
            val_loss += loss.item() * len(batch)
            
            # Per-sample errors
            mse = torch.mean((reconstruction - batch) ** 2, dim=(1, 2))
            val_errors.extend(mse.cpu().numpy())
    
    avg_val_loss = val_loss / len(X_val_tensor)
    val_losses.append(avg_val_loss)
    
    # Calculate AUC
    val_auc = roc_auc_score(y_val, val_errors)
    val_aucs.append(val_auc)
    
    # Learning rate scheduling
    scheduler.step(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), DATA_DIR / "bidirectional_full_best.pth")
        if (epoch + 1) % 5 == 0:
            print(f"✓ New best model (loss: {best_val_loss:.6f}, AUC: {val_auc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    # Progress
    if (epoch + 1) % 5 == 0:
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | Val AUC: {val_auc:.4f} | LR: {current_lr:.6f} | Time: {elapsed:.1f}s")

print(f"\n✅ Training complete! Time: {time.time() - start_time:.1f}s")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Best validation AUC: {max(val_aucs):.4f}")


TRAINING BIDIRECTIONAL LSTM (FULL LENGTH 986)
✓ New best model (loss: 0.887103, AUC: 0.5852)
Epoch [5/50] | Train Loss: 0.875737 | Val Loss: 0.887103 | Val AUC: 0.5852 | LR: 0.000300 | Time: 36.4s
✓ New best model (loss: 0.683584, AUC: 0.6544)
Epoch [10/50] | Train Loss: 0.678295 | Val Loss: 0.683584 | Val AUC: 0.6544 | LR: 0.000300 | Time: 70.8s
✓ New best model (loss: 0.604784, AUC: 0.7735)
Epoch [15/50] | Train Loss: 0.599411 | Val Loss: 0.604784 | Val AUC: 0.7735 | LR: 0.000300 | Time: 105.4s
✓ New best model (loss: 0.569994, AUC: 0.6070)
Epoch [20/50] | Train Loss: 0.554093 | Val Loss: 0.569994 | Val AUC: 0.6070 | LR: 0.000300 | Time: 140.1s
✓ New best model (loss: 0.504964, AUC: 0.7377)
Epoch [25/50] | Train Loss: 0.504939 | Val Loss: 0.504964 | Val AUC: 0.7377 | LR: 0.000300 | Time: 174.7s
Epoch [30/50] | Train Loss: 0.467515 | Val Loss: 0.570899 | Val AUC: 0.4965 | LR: 0.000300 | Time: 209.4s
Epoch [35/50] | Train Loss: 0.429939 | Val Loss: 0.522572 | Val AUC: 0.6741 | LR: 0.0

In [15]:
# ── BIDIRECTIONAL WITH FULL LENGTH + LOWER LEARNING RATE ─────────────────
# Reset and retrain with more conservative settings

# Re-initialize model
model = BidirectionalLSTMFull(
    input_dim=input_dim,
    hidden_dim=64,
    latent_dim=16,
    dropout=0.3
).to(device)

# Even lower learning rate for stability
optimizer = optim.Adam(model.parameters(), lr=0.0001)  # Was 0.0003
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

print("Retraining with LR=0.0001 for stability...")
print(f"Previous best AUC: 0.7758, targeting >0.78")

# Train for 40 epochs (it will early stop if needed)
num_epochs = 40
best_val_loss = float('inf')
patience = 8
patience_counter = 0

train_losses = []
val_losses = []
val_aucs = []

start_time = time.time()

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0
    for batch in train_loader:
        X_batch = batch[0]
        
        optimizer.zero_grad()
        reconstruction, _ = model(X_batch)
        loss = criterion(reconstruction, X_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation
    model.eval()
    val_loss = 0
    val_errors = []
    
    with torch.no_grad():
        for i in range(0, len(X_val_tensor), batch_size):
            batch = X_val_tensor[i:i+batch_size]
            reconstruction, _ = model(batch)
            loss = criterion(reconstruction, batch)
            val_loss += loss.item() * len(batch)
            
            mse = torch.mean((reconstruction - batch) ** 2, dim=(1, 2))
            val_errors.extend(mse.cpu().numpy())
    
    avg_val_loss = val_loss / len(X_val_tensor)
    val_losses.append(avg_val_loss)
    
    # Calculate AUC
    val_auc = roc_auc_score(y_val, val_errors)
    val_aucs.append(val_auc)
    
    # Learning rate scheduling
    scheduler.step(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), DATA_DIR / "bidirectional_full_stable.pth")
        if (epoch + 1) % 5 == 0:
            print(f"✓ New best (loss: {best_val_loss:.6f}, AUC: {val_auc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    # Progress
    if (epoch + 1) % 5 == 0:
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | Val AUC: {val_auc:.4f} | LR: {current_lr:.6f}")

print(f"\n Training complete!")
print(f"Best validation AUC: {max(val_aucs):.4f}")

Retraining with LR=0.0001 for stability...
Previous best AUC: 0.7758, targeting >0.78
✓ New best (loss: 0.989356, AUC: 0.4276)
Epoch [5/40] | Train Loss: 0.998505 | Val Loss: 0.989356 | Val AUC: 0.4276 | LR: 0.000100
✓ New best (loss: 0.882635, AUC: 0.5795)
Epoch [10/40] | Train Loss: 0.878147 | Val Loss: 0.882635 | Val AUC: 0.5795 | LR: 0.000100
✓ New best (loss: 0.726164, AUC: 0.6184)
Epoch [15/40] | Train Loss: 0.722819 | Val Loss: 0.726164 | Val AUC: 0.6184 | LR: 0.000100
✓ New best (loss: 0.687286, AUC: 0.6403)
Epoch [20/40] | Train Loss: 0.669973 | Val Loss: 0.687286 | Val AUC: 0.6403 | LR: 0.000100
Epoch [25/40] | Train Loss: 0.632011 | Val Loss: 0.669143 | Val AUC: 0.6900 | LR: 0.000100
✓ New best (loss: 0.639394, AUC: 0.6673)
Epoch [30/40] | Train Loss: 0.592682 | Val Loss: 0.639394 | Val AUC: 0.6673 | LR: 0.000100
Epoch [35/40] | Train Loss: 0.565334 | Val Loss: 0.687980 | Val AUC: 0.6922 | LR: 0.000100
✓ New best (loss: 0.584420, AUC: 0.6991)
Epoch [40/40] | Train Loss: 0.54

In [16]:
# ── EVALUATE YOUR BEST BIDIRECTIONAL MODEL (LR=0.0003, epoch ~25) ─────────
# Load the best model from your first bidirectional run
DATA_DIR = Path("data/processed")

# Check which model file exists
best_model_path = DATA_DIR / "bidirectional_full_best.pth"
if best_model_path.exists():
    print(f"Loading best bidirectional model from {best_model_path}")
    
    # Recreate the model architecture
    input_dim = 129  # Your number of sensors
    
    class BidirectionalLSTMFull(nn.Module):
        def __init__(self, input_dim, hidden_dim=64, latent_dim=16, dropout=0.3):
            super().__init__()
            self.encoder = nn.LSTM(input_dim, hidden_dim, 1, batch_first=True, bidirectional=True)
            self.fc_encode = nn.Linear(hidden_dim * 2, latent_dim)
            self.fc_decode = nn.Linear(latent_dim, hidden_dim)
            self.decoder = nn.LSTM(hidden_dim, hidden_dim, 1, batch_first=True)
            self.output = nn.Linear(hidden_dim, input_dim)
            self.dropout = nn.Dropout(dropout)
        
        def encode(self, x):
            lstm_out, _ = self.encoder(x)
            last_forward = lstm_out[:, -1, :self.encoder.hidden_size]
            last_backward = lstm_out[:, 0, self.encoder.hidden_size:]
            combined = torch.cat([last_forward, last_backward], dim=1)
            return self.fc_encode(combined)
        
        def decode(self, latent, seq_len):
            latent_exp = latent.unsqueeze(1).repeat(1, seq_len, 1)
            hidden = self.fc_decode(latent_exp)
            hidden = self.dropout(hidden)
            lstm_out, _ = self.decoder(hidden)
            return self.output(lstm_out)
        
        def forward(self, x):
            latent = self.encode(x)
            recon = self.decode(latent, x.shape[1])
            return recon, latent
    
    # Load model
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model = BidirectionalLSTMFull(input_dim=input_dim).to(device)
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()
    
    print("✓ Model loaded successfully")
    
else:
    print(f"Model not found at {best_model_path}")

Loading best bidirectional model from data/processed/bidirectional_full_best.pth
✓ Model loaded successfully


In [17]:
# ── GET TEST SET PERFORMANCE ──────────────────────────────────────────────
# Load test data
data = np.load(DATA_DIR / "sequences.npz")
X_test = data['X_test']
y_test = data['y_test']
X_val = data['X_val']
y_val = data['y_val']

def get_errors(model, data, device, batch_size=32):
    data_tensor = torch.FloatTensor(data).to(device)
    errors = []
    with torch.no_grad():
        for i in range(0, len(data), batch_size):
            batch = data_tensor[i:i+batch_size]
            reconstruction, _ = model(batch)
            mse = torch.mean((reconstruction - batch) ** 2, dim=(1, 2))
            errors.extend(mse.cpu().numpy())
    return np.array(errors)

print("Computing reconstruction errors...")
test_errors = get_errors(model, X_test, device)
val_errors = get_errors(model, X_val, device)

# Calculate metrics
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, roc_curve

test_auc = roc_auc_score(y_test, test_errors)
test_pr_auc = average_precision_score(y_test, test_errors)

print("\n" + "="*60)
print("BEST BIDIRECTIONAL MODEL - TEST SET PERFORMANCE")
print("="*60)
print(f"ROC-AUC: {test_auc:.4f}")
print(f"PR-AUC:  {test_pr_auc:.4f}")

# Find optimal threshold from validation
fpr, tpr, thresholds = roc_curve(y_val, val_errors)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print(f"\nOptimal threshold (from validation): {optimal_threshold:.4f}")

# Apply to test set
test_pred = (test_errors > optimal_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

print(f"\nConfusion Matrix:")
print(f"  True Negatives:  {tn}")
print(f"  False Positives: {fp}")
print(f"  False Negatives: {fn}")
print(f"  True Positives:  {tp}")

print(f"\nMetrics:")
print(f"  Precision: {tp/(tp+fp):.4f}")
print(f"  Recall:    {tp/(tp+fn):.4f}")
print(f"  F1-Score:  {2*tp/(2*tp+fp+fn):.4f}")
print(f"  Normal Recall: {tn/(tn+fp):.4f}")

Computing reconstruction errors...

BEST BIDIRECTIONAL MODEL - TEST SET PERFORMANCE
ROC-AUC: 0.6150
PR-AUC:  0.7361

Optimal threshold (from validation): 0.4113

Confusion Matrix:
  True Negatives:  112
  False Positives: 94
  False Negatives: 141
  True Positives:  237

Metrics:
  Precision: 0.7160
  Recall:    0.6270
  F1-Score:  0.6685
  Normal Recall: 0.5437


In [4]:
# ============================================================================
# CELL 3: SEQUENCE LENGTH SELECTION
# ============================================================================
# Calculate sample lengths
sample_lengths = df.groupby('sample').size()
print("Sample length statistics:")
print(f"  Min: {sample_lengths.min()}")
print(f"  Max: {sample_lengths.max()}")
print(f"  Mean: {sample_lengths.mean():.1f}")
print(f"  Median: {sample_lengths.median():.1f}")

# Choose sequence length (using last N timesteps for efficiency)
# You can adjust this: 986 (full), 500, 300
SEQ_LEN = 500  # Balanced choice - good memory and temporal coverage
print(f"\nUsing sequence length: {SEQ_LEN}")

# Check how many samples are long enough
samples_valid = (sample_lengths >= SEQ_LEN).sum()
print(f"Samples with at least {SEQ_LEN} timesteps: {samples_valid}/{len(sample_lengths)} ({samples_valid/len(sample_lengths)*100:.1f}%)")

Sample length statistics:
  Min: 986
  Max: 1164
  Mean: 1094.1
  Median: 1096.0

Using sequence length: 500
Samples with at least 500 timesteps: 2122/2122 (100.0%)


In [5]:
# ============================================================================
# CELL 4: CREATE TRAIN/VAL/TEST SPLITS (NO LEAKAGE)
# ============================================================================
# Create metadata
sample_meta = df.groupby('sample').agg(
    anomaly=('anomaly', 'first'),
    n_timesteps=('time', 'count')
).reset_index()

# Filter to samples with enough timesteps
sample_meta = sample_meta[sample_meta['n_timesteps'] >= SEQ_LEN].reset_index(drop=True)
print(f"After filtering: {len(sample_meta)} samples")

# Separate normal and anomaly
normal_samples = sample_meta[~sample_meta['anomaly']]
anomaly_samples = sample_meta[sample_meta['anomaly']]

print(f"\nNormal samples: {len(normal_samples)}")
print(f"Anomaly samples: {len(anomaly_samples)}")

# Split normal samples (70% train, 15% val, 15% test)
train_norm, temp_norm = train_test_split(normal_samples, test_size=0.30, random_state=42)
val_norm, test_norm = train_test_split(temp_norm, test_size=0.50, random_state=42)

# Split anomaly samples (50% val, 50% test)
val_anom, test_anom = train_test_split(anomaly_samples, test_size=0.50, random_state=42)

# Combine splits
train_samples = train_norm  # Only normal for training
val_samples = pd.concat([val_norm, val_anom])
test_samples = pd.concat([test_norm, test_anom])

print(f"\nFinal splits:")
print(f"  Train: {len(train_samples)} (all normal)")
print(f"  Val: {len(val_samples)} ({len(val_norm)} normal, {len(val_anom)} anomaly)")
print(f"  Test: {len(test_samples)} ({len(test_norm)} normal, {len(test_anom)} anomaly)")

After filtering: 2122 samples

Normal samples: 1367
Anomaly samples: 755

Final splits:
  Train: 956 (all normal)
  Val: 582 (205 normal, 377 anomaly)
  Test: 584 (206 normal, 378 anomaly)


In [6]:
# ============================================================================
# CELL 5: LOAD SEQUENCES (LAST SEQ_LEN TIMESTEPS)
# ============================================================================
def load_sequence(sample_id, df, sensors, seq_len):
    """Load last seq_len timesteps of sensor data"""
    sample_data = df[df['sample'] == sample_id][sensors].values
    # Take last seq_len timesteps
    if len(sample_data) >= seq_len:
        sequence = sample_data[-seq_len:]
    else:
        # Pad at beginning if needed (shouldn't happen due to filtering)
        pad_len = seq_len - len(sample_data)
        sequence = np.vstack([np.zeros((pad_len, len(sensors))), sample_data])
    return sequence

def build_sequences(sample_ids, df, sensors, seq_len):
    """Stack sequences into 3D array"""
    sequences = []
    for sid in sample_ids:
        seq = load_sequence(sid, df, sensors, seq_len)
        sequences.append(seq)
    return np.array(sequences)

print("Loading sequences...")
X_train_raw = build_sequences(train_samples['sample'], df, sensors_to_keep, SEQ_LEN)
X_val_raw = build_sequences(val_samples['sample'], df, sensors_to_keep, SEQ_LEN)
X_test_raw = build_sequences(test_samples['sample'], df, sensors_to_keep, SEQ_LEN)

# Labels
y_val = val_samples['anomaly'].astype(int).values
y_test = test_samples['anomaly'].astype(int).values

print(f"\nSequence shapes:")
print(f"  X_train: {X_train_raw.shape}")
print(f"  X_val: {X_val_raw.shape}")
print(f"  X_test: {X_test_raw.shape}")

Loading sequences...

Sequence shapes:
  X_train: (956, 500, 129)
  X_val: (582, 500, 129)
  X_test: (584, 500, 129)


In [7]:
# ============================================================================
# CELL 6: SCALE THE DATA
# ============================================================================
print("Scaling data...")

# Reshape to (N*T, F) for fitting
n_train, T, F = X_train_raw.shape
X_train_flat = X_train_raw.reshape(-1, F)

# Fit scaler on training data only
scaler = StandardScaler()
scaler.fit(X_train_flat)

# Transform all data
X_train = scaler.transform(X_train_flat).reshape(n_train, T, F).astype('float32')

n_val, T, F = X_val_raw.shape
X_val = scaler.transform(X_val_raw.reshape(-1, F)).reshape(n_val, T, F).astype('float32')

n_test, T, F = X_test_raw.shape
X_test = scaler.transform(X_test_raw.reshape(-1, F)).reshape(n_test, T, F).astype('float32')

print(f"Data scaled to {X_train.dtype}")
print(f"Mean: {X_train.mean():.4f}, Std: {X_train.std():.4f}")
print(f"Final shapes - Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Scaling data...
Data scaled to float32
Mean: -0.0000, Std: 1.0000
Final shapes - Train: (956, 500, 129), Val: (582, 500, 129), Test: (584, 500, 129)


In [8]:
# ============================================================================
# CELL 7: BIDIRECTIONAL LSTM AUTOENCODER MODEL
# ============================================================================
class BidirectionalLSTMEncoder(nn.Module):
    """
    Bidirectional LSTM Autoencoder for anomaly detection.
    Captures patterns from both past and future contexts.
    """
    def __init__(self, input_dim, hidden_dim=128, latent_dim=32, num_layers=2, dropout=0.2):
        super(BidirectionalLSTMEncoder, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.num_layers = num_layers
        
        # Encoder: Bidirectional LSTM
        self.encoder_lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Compress bidirectional output to latent space
        self.encoder_fc = nn.Linear(hidden_dim * 2, latent_dim)
        
        # Decoder: Regular LSTM
        self.decoder_fc = nn.Linear(latent_dim, hidden_dim)
        
        self.decoder_lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Output layer
        self.output_fc = nn.Linear(hidden_dim, input_dim)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
    def encode(self, x):
        """Encode input sequence to latent representation"""
        # Bidirectional encoding
        lstm_out, (hidden, cell) = self.encoder_lstm(x)
        
        # Take last timestep from both directions
        # For bidirectional: hidden shape is (2*num_layers, batch, hidden_dim)
        # We want the last forward and backward outputs
        batch_size = x.size(0)
        
        # Get last forward hidden state (from last layer)
        last_forward = lstm_out[:, -1, :self.hidden_dim]
        
        # Get first backward hidden state (from last layer)  
        last_backward = lstm_out[:, 0, self.hidden_dim:]
        
        # Concatenate
        last_hidden = torch.cat([last_forward, last_backward], dim=1)
        
        # Project to latent space
        latent = self.encoder_fc(last_hidden)
        
        return latent
    
    def decode(self, latent, seq_len):
        """Decode latent representation back to sequence"""
        # Expand latent to sequence length
        latent_expanded = latent.unsqueeze(1).repeat(1, seq_len, 1)
        
        # Project to hidden dimension
        hidden = self.decoder_fc(latent_expanded)
        hidden = self.dropout(hidden)
        
        # Decode with LSTM
        lstm_out, _ = self.decoder_lstm(hidden)
        
        # Generate reconstruction
        reconstruction = self.output_fc(lstm_out)
        
        return reconstruction
    
    def forward(self, x):
        """Forward pass: encode then decode"""
        seq_len = x.shape[1]
        latent = self.encode(x)
        reconstruction = self.decode(latent, seq_len)
        return reconstruction, latent

# Initialize model
input_dim = X_train.shape[2]
model = BidirectionalLSTMEncoder(
    input_dim=input_dim,
    hidden_dim=128,
    latent_dim=32,
    num_layers=2,
    dropout=0.2
).to(device)

print(f"Model architecture: Bidirectional LSTM Autoencoder")
print(f"Input dimension: {input_dim}")
print(f"Hidden dimension: 128")
print(f"Latent dimension: 32")
print(f"Number of layers: 2")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model on device: {next(model.parameters()).device}")

Model architecture: Bidirectional LSTM Autoencoder
Input dimension: 129
Hidden dimension: 128
Latent dimension: 32
Number of layers: 2
Total parameters: 953,761
Model on device: mps:0


In [9]:
# ============================================================================
# CELL 8: TRAINING SETUP
# ============================================================================
# Convert to PyTorch tensors
print("Moving data to device...")
X_train_tensor = torch.FloatTensor(X_train).to(device)
X_val_tensor = torch.FloatTensor(X_val).to(device)

# Create dataloader
batch_size = 16  # Adjust based on your memory
train_dataset = TensorDataset(X_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

print(f"Batch size: {batch_size}")
print(f"Training batches: {len(train_loader)}")
print(f"Learning rate: 0.0005")

Moving data to device...
Batch size: 16
Training batches: 60
Learning rate: 0.0005


In [10]:
# ============================================================================
# CELL 9: TRAINING LOOP
# ============================================================================
num_epochs = 50
best_val_loss = float('inf')
patience = 10
patience_counter = 0

train_losses = []
val_losses = []
val_aucs = []

print("\n" + "="*60)
print("STARTING BIDIRECTIONAL LSTM TRAINING")
print("="*60)

import time
start_time = time.time()

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0
    for batch in train_loader:
        X_batch = batch[0]
        
        optimizer.zero_grad()
        reconstruction, _ = model(X_batch)
        loss = criterion(reconstruction, X_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0
    val_errors = []
    
    with torch.no_grad():
        for i in range(0, len(X_val_tensor), batch_size):
            batch = X_val_tensor[i:i+batch_size]
            reconstruction, _ = model(batch)
            loss = criterion(reconstruction, batch)
            val_loss += loss.item() * len(batch)
            
            # Calculate per-sample reconstruction error
            mse = torch.mean((reconstruction - batch) ** 2, dim=(1, 2))
            val_errors.extend(mse.cpu().numpy())
    
    avg_val_loss = val_loss / len(X_val_tensor)
    val_losses.append(avg_val_loss)
    
    # Calculate validation AUC
    val_auc = roc_auc_score(y_val, val_errors)
    val_aucs.append(val_auc)
    
    # Learning rate scheduling
    scheduler.step(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), DATA_DIR / "bidirectional_lstm_best.pth")
        if (epoch + 1) % 5 == 0:
            print(f"✓ New best model saved (loss: {best_val_loss:.6f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    # Progress report
    if (epoch + 1) % 5 == 0:
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | Val AUC: {val_auc:.4f} | LR: {current_lr:.6f} | Time: {elapsed:.1f}s")

total_time = time.time() - start_time
print(f"\n✅ Training complete! Time: {total_time:.1f}s ({total_time/60:.1f} minutes)")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Best validation AUC: {max(val_aucs):.4f}")


STARTING BIDIRECTIONAL LSTM TRAINING
✓ New best model saved (loss: 0.824413)
Epoch [5/50] | Train Loss: 0.414753 | Val Loss: 0.824413 | Val AUC: 0.7848 | LR: 0.000500 | Time: 36.3s
✓ New best model saved (loss: 0.720680)
Epoch [10/50] | Train Loss: 0.275995 | Val Loss: 0.720680 | Val AUC: 0.7300 | LR: 0.000500 | Time: 73.1s
Epoch [15/50] | Train Loss: 0.234776 | Val Loss: 0.725580 | Val AUC: 0.7085 | LR: 0.000500 | Time: 110.0s
Epoch [20/50] | Train Loss: 0.248255 | Val Loss: 0.759336 | Val AUC: 0.6630 | LR: 0.000500 | Time: 146.8s
✓ New best model saved (loss: 0.669820)
Epoch [25/50] | Train Loss: 0.220944 | Val Loss: 0.669820 | Val AUC: 0.7752 | LR: 0.000250 | Time: 183.7s
Epoch [30/50] | Train Loss: 0.204932 | Val Loss: 0.663419 | Val AUC: 0.7801 | LR: 0.000250 | Time: 220.5s
Epoch [35/50] | Train Loss: 0.199823 | Val Loss: 0.708752 | Val AUC: 0.7257 | LR: 0.000250 | Time: 257.1s


KeyboardInterrupt: 